# Sequence Prediction - STRICT protocol

Strict copy of `_results_sequence_prediction.ipynb`. Every stage runs with
`--strict`, which switches the pipeline from our permissive default to the
evaluation population Screen2Vec itself uses in `modeltester.py`:

| | permissive (default notebook) | `--strict` (this notebook) |
|---|---|---|
| traces kept | >= 3 screens | >= `n_predictors + 1` (= 5) |
| context per prediction | 1..4 screens (truncated near trace start) | exactly 4 |
| candidate pool | every embedding in the directory | only screens of eligible traces |
| top-k% thresholds | `ceil()` | `int()` truncation, as in `modeltester.py:141` |
| training negatives | any screen | only screens of eligible traces |

Because every training window is full under `--strict`, the zero-padding
branch in the loss never fires, so training and evaluation see the same
context length. That mismatch is present in the permissive pipeline.

**Artefacts are suffixed `_strict`** and cannot clobber the permissive run:

    embeddings_seq_768_strict/                    (9_)
    lstm_predictor_768d_strict.pth                (9_)
    sequence_projector_768d_strict.pth            (9_)
    embeddings_seq_768_strict_with_app_desc/      (6_)

`6_generate_app_context_embeddings.py` needs no `--strict` of its own - it just
concatenates an app vector per screen, so it is pointed at the strict
directories instead.

### Two caveats on the numbers this produces

1. **`--strict` cannot reproduce Screen2Vec's screen *ordering*.** The original
   builds each trace from an unsorted `os.listdir()` of `view_hierarchies`
   (`UI_embedding/precompute_embeddings.py:72`), which is not temporal order and
   is not reproducible on another filesystem. This pipeline always uses
   `gestures.json` order, i.e. true temporal order.

2. **Screen-ID namespace bug, unresolved.** Embeddings are keyed by Rico's
   *global* `data/layouts/` numbering, while `gestures.json` keys are *per-trace*
   indices. They are different namespaces, so every trace is currently matched to
   unrelated screens (verified: 0 of 731 sampled trace screens match
   `layouts/<id>.json`). `--strict` does not fix this and inherits it. Treat the
   output below as protocol-comparable, not yet correct.

In [1]:
# Pipeline v2 - strict

In [2]:
%run 9_train_joint.py --proj_dim 768 --strict

Device: cpu
Joint training: Projector (4096 → 1024 → 768) + LSTM (768)
Loading 66261 embeddings from 'embeddings_4096_last' in parallel...


Loading embeddings: 100%|██████████| 66261/66261 [00:31<00:00, 2126.23it/s]


  Loaded matrix shape: (66261, 4096)
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')
Parsing 10292 traces in parallel...


Parsing traces: 100%|██████████| 10292/10292 [00:02<00:00, 5110.29it/s]


  Parsed 5095 traces
  STRICT: traces >= 5 screens, negative pool = 54520 of 66261 screens
  Input dim: 4096  |  Traces: 5095  |  Screens: 66261
  Train traces: 4586  |  Val traces: 509
  Projector parameters: 4,980,736
  LSTM parameters:      4,724,736
  Total parameters:     9,705,472

  Joint Training: 120 epochs, batch_size=256
  neg_samp=128, n_predictors=4
  temperature=0.07
  trace_negatives=context

Epoch   1/120 | Train: 5.2249 | Val: 3.5561 * best
Epoch   2/120 | Train: 3.3704 | Val: 2.9846 * best
Epoch   3/120 | Train: 2.8740 | Val: 2.7288 * best
Epoch   4/120 | Train: 2.6621 | Val: 2.6495 * best
Epoch   5/120 | Train: 2.5416 | Val: 2.6974 (patience 1/15)
Epoch   6/120 | Train: 2.4106 | Val: 2.6658 (patience 2/15)
Epoch   7/120 | Train: 2.3006 | Val: 2.6101 * best
Epoch   8/120 | Train: 2.2462 | Val: 2.5641 * best
Epoch   9/120 | Train: 2.1553 | Val: 2.6166 (patience 1/15)
Epoch  10/120 | Train: 2.1003 | Val: 2.5168 * best
Epoch  11/120 | Train: 2.0711 | Val: 2.5587 (patienc

Projecting: 100%|██████████| 17/17 [00:02<00:00,  6.60it/s]


  Projected shape: (66261, 768)


Saving embeddings: 100%|██████████| 66261/66261 [01:21<00:00, 814.97it/s] 


Saved LSTM predictor: lstm_predictor_768d_strict.pth
Saved projector: sequence_projector_768d_strict.pth
Saved training plot: joint_training_768d_strict.png

Total training time: 1827.8s

Next steps:
  python 6_generate_app_context_embeddings.py \
    --embeddings_dir embeddings_seq_768_strict \
    --output_dir embeddings_seq_768_strict_with_app

  python 4_evaluate_screen_prediction.py \
    --embeddings_dir embeddings_seq_768_strict_with_app \
    --lstm_model lstm_predictor_768d_strict.pth \
    --strict


In [3]:
%run 6_generate_app_context_embeddings.py --embeddings_dir embeddings_seq_768_strict --output_dir embeddings_seq_768_strict_with_app_desc

C:\Users\Jonathan\Dev\screen2vec-revisited\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading app details from ../rico-data/data/play_store_metadata/app_details_enriched.csv...
  CSV columns: ['App Package Name', 'Play Store Name', 'Category', 'Average Rating', 'Number of Ratings', 'Number of Downloads', 'Date Updated', 'Icon URL', 'gp_title', 'gp_description', 'gp_descriptionHTML', 'gp_summary', 'gp_installs', 'gp_minInstalls', 'gp_realInstalls', 'gp_score', 'gp_ratings', 'gp_reviews', 'gp_histogram', 'gp_price', 'gp_free', 'gp_currency', 'gp_sale', 'gp_saleTime', 'gp_originalPrice', 'gp_saleText', 'gp_offersIAP', 'gp_inAppProductPrice', 'gp_developer', 'gp_developerId', 'gp_developerEmail', 'gp_developerWebsite', 'gp_developerAddress', 'gp_privacyPolicy', 'gp_genre', 'gp_genreId', 'gp_categories', 'gp_icon', 'gp_headerImage', 'gp_screenshots', 'gp_video', 'gp_videoImage', 'gp_contentRating', 'gp_contentRatingDescription', 'gp_adSupported', 'gp_containsAds', 'gp_released', 'gp_lastUpdatedOn', 'gp_updated', 'gp_version', 'gp_comments', 'gp_status']
  Found description c

Parsing layout JSONs: 100%|██████████| 104/104 [00:00<00:00, 854.39it/s]


Preparing text input for embedding model...
Loading SBERT model 'sentence-transformers/all-mpnet-base-v2' (embed_dim=768)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4123.11it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding app packages...


... (2 outputs left out)

Saving concatenated embeddings:   0%|          | 28/66261 [00:00<03:58, 277.17it/s]

Concatenating screen (768-dim) + app (768-dim) embeddings...
Output dimension: 1536-dim


Saving concatenated embeddings: 100%|██████████| 66261/66261 [02:42<00:00, 407.59it/s]


Success! Concatenated embeddings created and saved.


In [4]:
%run 4_evaluate_screen_prediction.py --embeddings_dir embeddings_seq_768_strict_with_app_desc --lstm_model lstm_predictor_768d_strict.pth --strict


[Step 0/3] Loading LSTM predictor from 'lstm_predictor_768d_strict.pth'...
  Loaded LSTM predictor (embed_dim=768, best_val_loss=2.5168)

[Step 1/3] Loading embeddings...
Loading 66261 embeddings from 'embeddings_seq_768_strict_with_app_desc' in parallel...


Loading embeddings: 100%|██████████| 66261/66261 [00:43<00:00, 1528.80it/s]


  Loaded matrix shape: (66261, 1536)
  66261 screen embeddings available

[Step 2/3] Parsing RICO interaction traces...
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')


Parsing traces: 100%|██████████| 9384/9384 [00:09<00:00, 1025.53it/s]


  Parsed 5095 traces (54520 screens)
  Skipped 5197 traces with < 5 screens
  Skipped 104 screens absent from the trace->layout index
  Strict pool: restricted to 54520 screens from eligible traces (was 66261)

[Step 3/3] Running screen prediction evaluation...

  Protocol: STRICT (Screen2Vec modeltester.py population, exactly 4 context screens per prediction)

  Total screens in embedding index: 54520
  Top-0.01% threshold: k=5
  Top-0.1%  threshold: k=54
  Top-1%    threshold: k=545
  Top-5%    threshold: k=2726
  Mean embedding L2 norm: 1.4142
  Prediction mode: LSTM (Screen2Vec-equivalent)
  Evaluating traces in parallel via threads (n_jobs=-1)...


Evaluating traces (LSTM): 100%|██████████| 5095/5095 [56:49<00:00,  1.49it/s] 



  Total prediction instances: 34140

  GUI Screen Prediction — Comparison with Paper Table 3
Model                                            Top-1  Top-0.01%   Top-0.1%   Top-1%   Top-5%  Norm.RMSE
---------------------------------------------------------------------------------------------------------
>>> Qwen3-VL + LSTM (Ours) [strict] <<<         0.1182     0.5004     0.9917   0.9994   1.0000     0.6028
---------------------------------------------------------------------------------------------------------
Screen2Vec-EUCLIDEAN-spatial                    0.0610     0.2580     0.9690   0.9980   1.0000     0.8530
Screen2Vec-EUCLIDEAN-no spatial                 0.0065     0.1160     0.8960   0.9860   0.9990     1.7230
TextOnly                                        0.0120     0.0550     0.1960   0.4390   0.6430     1.2410
LayoutOnly                                      0.0041     0.0240     0.0910   0.2220   0.3950     1.1350
VisualOnly                                      0.0060    